# 02 EDA and Featurization

This notebook summarizes the processed 15-5PH fastener dataset, creates the proof-of-concept target/property figure, and defines the domain-derived feature set used for modeling.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Make paths work whether the notebook is run from the repo root or notebooks folder
NOTEBOOK_DIR = Path.cwd()
if NOTEBOOK_DIR.name == "notebooks":
    REPO_ROOT = NOTEBOOK_DIR.parent
else:
    REPO_ROOT = NOTEBOOK_DIR

DATA_DIR = REPO_ROOT / "data"
FIGURE_DIR = REPO_ROOT / "figures"
FIGURE_DIR.mkdir(exist_ok=True)

processed_path = DATA_DIR / "processed_material_database.csv"


## Load processed dataset

In [ ]:
df = pd.read_csv(processed_path)

print("Dataset shape:")
print(df.shape)

display(df.head())


Target property summary

The target property is job-level mean double-shear strength in ksi. The dataset is small, so these summaries are used to describe the proof-of-concept data rather than to make strong statistical claims.

In [ ]:
target_col = "MeanShear_ksi"

print("Target property: MeanShear_ksi (ksi)")
display(df[target_col].describe())

print("\nAge temperatures:")
display(df["AgeTemp_F"].value_counts().sort_index())

print("\nSpecimen counts by job/test stage:")
display(df[["JobNum", "Lot_ID", "TestStage", "SpecimenCount", "MeanShear_ksi"]])


## Figure 1: shear strength versus age temperature

In [ ]:
plt.figure(figsize=(7, 5))

plt.scatter(
    df["AgeTemp_F"],
    df["MeanShear_ksi"],
    s=80
)

for _, row in df.iterrows():
    label = f'{row["TestStage"]}'
    plt.text(
        row["AgeTemp_F"],
        row["MeanShear_ksi"],
        label,
        fontsize=9,
        ha="left",
        va="bottom"
    )

plt.xlabel("Age temperature (°F)")
plt.ylabel("Mean double-shear strength (ksi)")
plt.title("Job-level mean shear strength vs. age temperature")
plt.grid(True, alpha=0.3)

plt.tight_layout()
fig_path = FIGURE_DIR / "figure1_shear_vs_age_temp.png"
plt.savefig(fig_path, dpi=150, bbox_inches="tight")
plt.show()

print(f"Saved figure to: {fig_path}")


## Featurization

The feature set uses domain-derived manufacturing descriptors. These include heat-treatment variables, measured geometry, hardness, certification mechanical properties, and alloy chemistry. Net cross-sectional area is included because shear load depends directly on the area.

In [ ]:
numeric_features = [
    "AgeTemp_F",
    "AgeTime_hr",
    "HeatTreatSequence",
    "Reaged",
    "Mean_OD_in",
    "Mean_ID_in",
    "MeanNetArea_in2",
    "Mean_HRC",
    "SpecimenCount",
    "C", "Mn", "Si", "P", "S", "Ni", "Cr", "Mo", "Cu", "Nb",
    "RawDiameter",
    "SolutionTemp_F",
    "SolutionTime_hr",
    "CertAgeTemp_F",
    "CertAgeTime_hr",
    "SolutionHard_HBW",
    "Cert_UTS_ksi",
    "Cert_Yield_ksi",
    "Cert_Elong_pct",
    "Cert_RA_pct",
    "GrainSize"
]

categorical_features = ["TestStage"]

# Keep only features that exist and are not entirely missing
numeric_features = [
    c for c in numeric_features
    if c in df.columns and not df[c].isna().all()
]

categorical_features = [
    c for c in categorical_features
    if c in df.columns and not df[c].isna().all()
]

print("Numeric features:")
print(numeric_features)

print("\nCategorical features:")
print(categorical_features)

print("\nFeatures removed because missing or unavailable:")
all_candidate_features = set([
    "AgeTemp_F", "AgeTime_hr", "HeatTreatSequence", "Reaged",
    "Mean_OD_in", "Mean_ID_in", "MeanNetArea_in2", "Mean_HRC",
    "SpecimenCount", "C", "Mn", "Si", "P", "S", "Ni", "Cr", "Mo", "Cu", "Nb",
    "RawDiameter", "SolutionTemp_F", "SolutionTime_hr", "CertAgeTemp_F",
    "CertAgeTime_hr", "SolutionHard_HBW", "Cert_UTS_ksi", "Cert_Yield_ksi",
    "Cert_Elong_pct", "Cert_RA_pct", "GrainSize", "TestStage"
])
used_features = set(numeric_features + categorical_features)
print(sorted(all_candidate_features - used_features))


## Save feature list for later notebooks

In [ ]:
feature_info = pd.DataFrame({
    "feature": numeric_features + categorical_features,
    "type": ["numeric"] * len(numeric_features) + ["categorical"] * len(categorical_features)
})

feature_info_path = DATA_DIR / "feature_list.csv"
feature_info.to_csv(feature_info_path, index=False)

display(feature_info)
print(f"Saved feature list to: {feature_info_path}")
